# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and exploration of the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [`mlcroissant`](https://mlcommons.org/croissant/) Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dataclass, not a dict
print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Authors: {[author for author in metadata.author] if hasattr(metadata, 'author') else 'N/A'}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

- **Record Sets** are collections of records in the dataset (e.g., the tabular data tables, versioned by `@id`).
- **Fields** are individual data columns/attributes within those record sets, each with a unique `@id`.
- We will fetch the available record sets and list their fields and columns by their `@id`.

**Note:** All references (for record sets, fields, and columns) are with the explicit `@id` per the Croissant schema mapping.

In [ ]:
# List all record sets, field IDs, and columns
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    if rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field name: {f.name}, Field @id: {f.id}, DataType: {f.data_type}")
    print()

# Example: List a few records from the first record set by @id
if record_set_ids:
    preview_rs_id = record_set_ids[0]
    print(f"Previewing first 3 records of RecordSet: {preview_rs_id}\n")
    for ix, record in enumerate(dataset.records(record_set=preview_rs_id)):
        print(json.dumps(record, indent=2))
        if ix >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using the record set and field `@id`s from above.

We will loop through all found `@id`s for record sets and create a DataFrame.

In [ ]:
# Extract data from each available record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet: {record_set_id}, shape: {df.shape}")
        if not df.empty:
            print("  Columns:", df.columns.tolist())
            print(df.head(2))
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

# Pick the main tabular record set for further processing (usually the largest)
main_record_set_id = record_set_ids[0]  # Here it's the first, adjust as needed
print(f"\nMain record set selected: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Typical tasks include:
- Filtering records based on criteria
- Normalizing numeric fields
- Grouping data by categorical variables

> **Tip:** All field/column references in code use their `@id` (they look like full URIs or unique names), as per the Croissant schema.

Let's select a numeric field (e.g., GAD-7 score) and a group field (e.g., `level_of_education` or similar demographic).

In [ ]:
df = dataframes[main_record_set_id]

# Find a numeric field (typically has float/integer values)
# Example: 'gad7_score' or similar, match exact column name as it appears
print("Columns available in chosen record set:")
print(df.columns.tolist())

# Set the field IDs to use (adjust if your schema uses different IDs)
gad7_field_id = None
education_field_id = None
for col in df.columns:
    if 'gad' in col.lower() and 'score' in col.lower():
        gad7_field_id = col
    if 'educ' in col.lower():
        education_field_id = col

print(f"Numeric field for EDA: {gad7_field_id}")
print(f"Grouping field: {education_field_id}\n")

if gad7_field_id and gad7_field_id in df.columns:
    # Ensure the field is numeric
    df[gad7_field_id] = pd.to_numeric(df[gad7_field_id], errors='coerce')

    threshold = 7
    filtered_df = df[df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold} (N={len(filtered_df)}):")
    print(filtered_df[[gad7_field_id]].head())

    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"\nNormalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    if education_field_id and education_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(education_field_id)[gad7_field_id].mean().reset_index()
        print(f"\nMean GAD-7 score by education level:")
        print(grouped_df.head())
else:
    print("No suitable GAD-7 score field found for numeric EDA.")

## 5. Visualization
Let's visualize the distribution of GAD-7 scores, and optionally, GAD-7 scores by education group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id and gad7_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {gad7_field_id}")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    if education_field_id and education_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=education_field_id, x=gad7_field_id, data=df)
        plt.title(f"{gad7_field_id} by {education_field_id}")
        plt.xlabel("GAD-7 Score")
        plt.ylabel("Level of Education")
        plt.show()
else:
    print("No visualization possible; numeric GAD-7 field not found.")

## 6. Conclusion
In this notebook, we demonstrated loading the Kilifi County Mental Health Survey dataset from Croissant, inspecting record set and field structure by `@id`, extracting tabular data using `mlcroissant`, and performing simple EDA and visualization. All data elements are referenced via their Croissant `@id` for reproducibility and schema clarity.

**Key Findings:**
- Record set(s) and fields are uniquely referenced via their `@id` in Croissant.
- GAD-7 scores can be used to identify participants with higher anxiety levels and further analyzed by demographic fields such as education.
- The dataset is AI-ready thanks to its FAIR and Croissant-conformant structure, facilitating downstream use in research and machine learning.

> For advanced analysis, consider exploring other mental health assessment scores and additional demographic/comorbidity fields referenced by their `@id`s in the dataset.